In [153]:
import pandas as pd
import numpy as np

In [154]:
df = pd.read_csv("OnlineRetail.csv", encoding_errors="ignore")

### Data Understanding

In [155]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [156]:
df.tail()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12/9/2011 12:50,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/2011 12:50,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/2011 12:50,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/2011 12:50,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,12/9/2011 12:50,4.95,12680.0,France


In [157]:
df.shape

(541909, 8)

In [158]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [159]:
df["Country"].unique()

array(['United Kingdom', 'France', 'Australia', 'Netherlands', 'Germany',
       'Norway', 'EIRE', 'Switzerland', 'Spain', 'Poland', 'Portugal',
       'Italy', 'Belgium', 'Lithuania', 'Japan', 'Iceland',
       'Channel Islands', 'Denmark', 'Cyprus', 'Sweden', 'Austria',
       'Israel', 'Finland', 'Bahrain', 'Greece', 'Hong Kong', 'Singapore',
       'Lebanon', 'United Arab Emirates', 'Saudi Arabia',
       'Czech Republic', 'Canada', 'Unspecified', 'Brazil', 'USA',
       'European Community', 'Malta', 'RSA'], dtype=object)

In [160]:
df["Country"].value_counts()

Country
United Kingdom          495478
Germany                   9495
France                    8557
EIRE                      8196
Spain                     2533
Netherlands               2371
Belgium                   2069
Switzerland               2002
Portugal                  1519
Australia                 1259
Norway                    1086
Italy                      803
Channel Islands            758
Finland                    695
Cyprus                     622
Sweden                     462
Unspecified                446
Austria                    401
Denmark                    389
Japan                      358
Poland                     341
Israel                     297
USA                        291
Hong Kong                  288
Singapore                  229
Iceland                    182
Canada                     151
Greece                     146
Malta                      127
United Arab Emirates        68
European Community          61
RSA                         58


In [161]:
# filter out United Kingdom and EIRE rows, keeping all other countries
df_filtered = df[(df["Country"] != "United Kingdom") & (df["Country"] != "EIRE")]

In [162]:
df_filtered.shape

(38235, 8)

Handle missing values in `CustomerID` 

In [163]:
# assess missing values in CustomerID
missing = df_filtered["CustomerID"].isnull()
count_missing = missing.sum()

total = len(df)
pct_missing = (count_missing / total) * 100 

print(f"Missing CustomerIDs: {missing}")
print(f"Missing percentage: {pct_missing:.2f}%")

Missing CustomerIDs: 26        False
27        False
28        False
29        False
30        False
          ...  
541904    False
541905    False
541906    False
541907    False
541908    False
Name: CustomerID, Length: 38235, dtype: bool
Missing percentage: 0.14%


In [164]:
# drop rows with missing CustomerID - only 0.14% of the dataset
df_filtered = df_filtered.dropna(subset="CustomerID")

In [165]:
# sanity check - should print 0 
missing = df_filtered["CustomerID"].isnull()
count_missing = missing.sum()

print(count_missing)

0


Assess cancellations using `InvoiceNo`

In [166]:
df["InvoiceNo"].sample(40)

284143    561860
20787     538071
314412    564635
257822    559545
14615     537606
125667    547055
466157    576321
327008    565617
282117    561626
164628    550651
221330    556243
382323    569898
280893    561507
249007    558877
339707    566602
395662    571039
529079    580730
180665    552333
15168     537638
487018    577768
224501    556529
445811    574863
283801    561804
134843    547870
426153    573345
507043    579167
128503    547350
362848    568527
238055    557894
298911    563073
446686    574914
505845    579101
366573    568780
72160     542231
459199    575868
323326    565289
502889    578852
287967    562123
164326    550638
311884    564327
Name: InvoiceNo, dtype: object

In [167]:
num_cancelled = df_filtered["InvoiceNo"].astype(str).str.startswith("C").sum()
print(num_cancelled)

1125


In [168]:
pct_cancelled = (num_cancelled / total) * 100
print(pct_cancelled)

0.2075994309007601


In [169]:
# drop rows with cancelled orders
mask =  df_filtered["InvoiceNo"].astype(str).str.startswith("C")
df_filtered = df_filtered[~mask]

In [170]:
df_filtered.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
26,536370,22728,ALARM CLOCK BAKELIKE PINK,24,12/1/2010 8:45,3.75,12583.0,France
27,536370,22727,ALARM CLOCK BAKELIKE RED,24,12/1/2010 8:45,3.75,12583.0,France
28,536370,22726,ALARM CLOCK BAKELIKE GREEN,12,12/1/2010 8:45,3.75,12583.0,France
29,536370,21724,PANDA AND BUNNIES STICKER SHEET,12,12/1/2010 8:45,0.85,12583.0,France
30,536370,21883,STARS GIFT TAPE,24,12/1/2010 8:45,0.65,12583.0,France


Investigate non-numeric `StockCode`s

In [171]:
df_filtered["StockCode"].sample(40)

77169      22449
312782     20679
285254     21683
378408     23388
495296     84988
212269     21931
122187    84997D
19576      22690
275621     21238
116561     21669
277788     21121
156437     21531
209153     84755
377674     22616
342091     23222
11476      22411
59692      22629
152483    35810A
287048     21205
121733     22960
154890     22305
531900     23256
309331     23207
71557      20724
182390     20726
364131     22809
531428     21086
364172     21533
353152     23295
85680      23231
485803     23598
61927      20750
480697     23296
200575     21705
323042     21883
215426     21394
25782     84520B
130491     22699
469772     23084
129351     22961
Name: StockCode, dtype: object

In [172]:
# inspect and count the number of non-numeric StockCodes
mask = ~df_filtered["StockCode"].astype(str).str.match(r"^\d+[A-Z]?$")
df_filtered["StockCode"][mask].value_counts()

StockCode
POST       1072
15056BL      49
M            37
C2            5
Name: count, dtype: int64

In [173]:
invalid_stockcodes = ["POST", "M", "C2"]
mask = df_filtered["StockCode"].isin(invalid_stockcodes)
df_filtered = df_filtered[~mask]

Fix datatypes

In [174]:
df_filtered.dtypes

InvoiceNo       object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object

In [175]:
df_filtered["InvoiceDate"] = pd.to_datetime(df_filtered["InvoiceDate"], format="%m/%d/%Y %H:%M")

In [176]:
df_filtered["CustomerID"] = df_filtered["CustomerID"].astype(str)

In [177]:
df_filtered.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID             object
Country                object
dtype: object

In [182]:
df_filtered.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
26,536370,22728,ALARM CLOCK BAKELIKE PINK,24,2010-12-01 08:45:00,3.75,12583.0,France
27,536370,22727,ALARM CLOCK BAKELIKE RED,24,2010-12-01 08:45:00,3.75,12583.0,France
28,536370,22726,ALARM CLOCK BAKELIKE GREEN,12,2010-12-01 08:45:00,3.75,12583.0,France
29,536370,21724,PANDA AND BUNNIES STICKER SHEET,12,2010-12-01 08:45:00,0.85,12583.0,France
30,536370,21883,STARS GIFT TAPE,24,2010-12-01 08:45:00,0.65,12583.0,France


In [183]:
df_filtered.tail()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
541908,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [184]:
df_filtered.shape

(35227, 8)

In [185]:
df_filtered.to_csv('clean_data.csv', index=False)

In [186]:
df_filtered.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID             object
Country                object
dtype: object